In [0]:
pip install lyricsgenius

In [0]:
GENIUS_ACCESS_TOKEN="GhMy7RHHHGbIGNtZxQGZ6G1z3X5xF3R4OLAxfCH_OzOYtxMHmzpMDVw4efcKF8rC"

In [0]:
import os
import lyricsgenius


genius = lyricsgenius.Genius(
    GENIUS_ACCESS_TOKEN,
    timeout=15,
    retries=3,
    remove_section_headers=True,
    skip_non_songs=True,
 )
genius.verbose = False

In [0]:
def fetch_lyrics_genius(client, title: str, artist: str) -> str:
    try:
        hit = client.search_song(title=title, artist=artist)
        if hit and hit.lyrics:
            return hit.lyrics.strip()
    except Exception as e:
        print(f"[warn] {title!r} by {artist!r}: {e}")
    return ""

In [0]:
# STEP 1: Search Title using spotify_uri from titles.csv
spotify_uri = "1OHj2WRXOR9XdCV6PuptXv"
titles_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"))
matching_row = titles_df[titles_df["spotify_uri"] == spotify_uri]
song_title = matching_row["title"].iloc[0] if not matching_row.empty else None
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else None
print(f"\nSpotify URI: {spotify_uri}")
print(f"Title: {song_title}")
print(f"Artist: {song_artist}")

In [0]:
# STEP 1: Search Lyrics using Title
song_title = "Amarte más no pude - En vivo"
song_artist = ""


In [0]:
# STEP 2: Fetch lyrics using Genius API
lyrics = fetch_lyrics_genius(genius, song_title, song_artist)

print("\nLyrics snippet:\n")
print((lyrics or "<no lyrics found>")[:1200])

In [0]:
# STEP 2: Get spotify_uri from titles.csv by title and artist
from pathlib import Path
import pandas as pd

if song_artist != "":
    matching_row = titles_df[
        titles_df["title"].str.contains(song_title, case=False, na=False) &
        titles_df["artist"].str.contains(song_artist, case=False, na=False)
    ]
else:
    matching_row = titles_df[
        titles_df["artist"].str.contains(song_title, case=False, na=False)
    ]

song_uri = matching_row["spotify_uri"].iloc[0] if not matching_row.empty else None
song_title = matching_row["title"].iloc[0] if not matching_row.empty else song_title
song_artist = matching_row["artist"].iloc[0] if not matching_row.empty else song_artist

print(f"\nSpotify URI for '{song_title}' by '{song_artist}': {song_uri or '<not found>'}")

In [0]:
# STEP 3: Update titles.csv with song_title and song_artist for the row with the matching spotify_uri (if found)
if song_uri:
    titles_df.loc[titles_df["spotify_uri"] == song_uri, ["title", "artist"]] = [song_title, song_artist]
    titles_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/titles/2026/03/05/titles.csv"), index=False)
    print(f"Updated titles.csv with title '{song_title}' and artist '{song_artist}' for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri found for title '{song_title}' and artist '{song_artist}'. No updates made to titles.csv.")

In [0]:
# STEP 4: Update lyrics.csv with lyrics for the matching spotify_uri
lyrics_df = pd.read_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"))
if song_uri and lyrics:
    lyrics_df.loc[lyrics_df["spotify_uri"] == song_uri, "lyrics"] = lyrics
    lyrics_df.to_csv(Path("/Users/wednesday/Documents/GitHub/lyrics_analysis/data/processed/lyrics/2026/03/05/lyrics.csv"), index=False)
    print(f"Updated lyrics.csv with lyrics for spotify_uri '{song_uri}'.")
else:
    print(f"No matching spotify_uri or lyrics found for title '{song_title}' and artist '{song_artist}'. No updates made to lyrics.csv.")